In [8]:
!wget http://www.manythings.org/anki/rus-eng.zip
!unzip -o rus-eng.zip

--2026-04-07 10:58:31--  http://www.manythings.org/anki/rus-eng.zip
Resolving www.manythings.org (www.manythings.org)... 173.254.30.110
Connecting to www.manythings.org (www.manythings.org)|173.254.30.110|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17447329 (17M) [application/zip]
Saving to: ‘rus-eng.zip.1’

rus-eng.zip.1       100%[===================>]  16.64M  19.1MB/s    in 0.9s    

2026-04-07 10:58:33 (19.1 MB/s) - ‘rus-eng.zip.1’ saved [17447329/17447329]

Archive:  rus-eng.zip
  inflating: rus.txt                 
  inflating: _about.txt              


In [9]:
import re
import math
import time
import random
from collections import Counter
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

# 1. Read and clean data
file_path = "rus.txt"
pairs = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            pairs.append((parts[0], parts[1]))

# --- DATA CLIPPING ---
# Keep only the first 50,000 sentence pairs to massively speed up training.
# You can change this number to 10000 for even faster testing, or 100000 for better accuracy.
pairs = pairs[:50000]
# ---------------------

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zа-яё\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

processed_pairs = []
for eng, rus in pairs:
    eng = clean_text(eng)
    rus = "<sos> " + clean_text(rus) + " <eos>"
    processed_pairs.append((eng, rus))

# 2. Build Vocabulary
def build_vocab(sentences):
    counter = Counter()
    for sent in sentences:
        counter.update(sent.split())
    vocab = {"<pad>": 0, "<oov>": 1}
    for word in counter:
        vocab[word] = len(vocab)
    return vocab

eng_vocab = build_vocab([p[0] for p in processed_pairs])
rus_vocab = build_vocab([p[1] for p in processed_pairs])

# 3. Convert to Indices and Split
def sentence_to_idx(sentence, vocab):
    if isinstance(sentence, str):
        sentence = sentence.split()
    return [vocab.get(word, vocab["<oov>"]) for word in sentence]

data = []
for eng, rus in processed_pairs:
    data.append((sentence_to_idx(eng, eng_vocab), sentence_to_idx(rus, rus_vocab)))

# Split numerical data for the DataLoader
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)

# Split raw text pairs using the exact same random state for our later BLEU evaluation
_, val_text_pairs = train_test_split(processed_pairs, test_size=0.2, random_state=42)

print(f"Training data size: {len(train_data)}")
print(f"Validation data size: {len(val_data)}")
print(f"English vocab size: {len(eng_vocab)}")
print(f"Russian vocab size: {len(rus_vocab)}")

Training data size: 40000
Validation data size: 10000
English vocab size: 4932
Russian vocab size: 13768


In [10]:
class TranslationDataset(Dataset):
    def __init__(self, sentence_pairs):
        self.pairs = sentence_pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        eng, rus = self.pairs[idx]
        return torch.tensor(eng), torch.tensor(rus)

def collate_fn(batch):
    eng_batch, rus_batch = [], []
    for eng, rus in batch:
        eng_batch.append(eng)
        rus_batch.append(rus)

    eng_padded = pad_sequence(eng_batch, padding_value=0)
    rus_padded = pad_sequence(rus_batch, padding_value=0)
    return eng_padded, rus_padded

batch_size = 32

train_dataset = TranslationDataset(train_data)
val_dataset = TranslationDataset(val_data)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

In [11]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout if n_layers > 1 else 0)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs, mask=None):
        src_len = encoder_outputs.shape[0]
        hidden = hidden[-1].unsqueeze(1).repeat(1, src_len, 1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e10)
        return F.softmax(attention, dim=1)

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, attention, dropout=0.5):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim + hidden_dim, hidden_dim)
        self.fc_out = nn.Linear(emb_dim + hidden_dim + hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs, mask=None):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        a = self.attention(hidden, encoder_outputs, mask).unsqueeze(1)
        context = torch.bmm(a, encoder_outputs.permute(1, 0, 2)).permute(1, 0, 2)
        output, (hidden, cell) = self.rnn(torch.cat((embedded, context), dim=2), (hidden, cell))
        prediction = self.fc_out(torch.cat((output.squeeze(0), context.squeeze(0), embedded.squeeze(0)), dim=1))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5, mask=None):
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        input = trg[0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs, mask)
            outputs[t] = output
            top1 = output.argmax(1)
            input = trg[t] if random.random() < teacher_forcing_ratio else top1

        return outputs

def init_weights(m):
    for name, param in m.named_parameters():
        if 'weight' in name:
            nn.init.normal_(param.data, mean=0, std=0.01)
        else:
            nn.init.constant_(param.data, 0)

In [12]:
# Instantiate Model
input_dim  = len(eng_vocab)
output_dim = len(rus_vocab)
emb_dim    = 256
hidden_dim = 512
n_layers   = 1
dropout    = 0.5

attention = Attention(hidden_dim)
encoder   = Encoder(input_dim, emb_dim, hidden_dim, n_layers, dropout)
decoder   = Decoder(output_dim, emb_dim, hidden_dim, attention, dropout)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = Seq2Seq(encoder, decoder, device).to(device)
model.apply(init_weights)

optimizer = optim.Adam(model.parameters(), lr=0.0005)
pad_idx   = rus_vocab["<pad>"]
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

def train(model, loader, optimizer, criterion, clip, device):
    model.train()
    epoch_loss = 0
    tf_ratio = 0.5
    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio=tf_ratio)
        output_dim = output.shape[-1]
        output  = output[1:].view(-1, output_dim)
        trg_flat = trg[1:].view(-1)
        loss = criterion(output, trg_flat)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, trg in loader:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg, teacher_forcing_ratio=0)
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg_flat = trg[1:].view(-1)
            loss = criterion(output, trg_flat)
            epoch_loss += loss.item()
    return epoch_loss / len(loader)

# The Training Loop
clip = 1
num_epochs = 10
best_val_loss = float('inf')

print("Starting training...")
for epoch in range(num_epochs):
    train_loss = train(model, train_loader, optimizer, criterion, clip, device)
    val_loss   = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_att_model_final.pt')

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

# Load the best model before moving to evaluation
model.load_state_dict(torch.load('best_att_model_final.pt', map_location=device))
model.eval()

Starting training...
Epoch 01 | Train Loss: 4.9140 | Val Loss: 4.4062 | LR: 0.000500
Epoch 02 | Train Loss: 3.7883 | Val Loss: 3.7472 | LR: 0.000500
Epoch 03 | Train Loss: 2.9579 | Val Loss: 3.2941 | LR: 0.000500
Epoch 04 | Train Loss: 2.2843 | Val Loss: 3.0012 | LR: 0.000500
Epoch 05 | Train Loss: 1.7733 | Val Loss: 2.8241 | LR: 0.000500
Epoch 06 | Train Loss: 1.4126 | Val Loss: 2.7473 | LR: 0.000500
Epoch 07 | Train Loss: 1.1718 | Val Loss: 2.7169 | LR: 0.000500
Epoch 08 | Train Loss: 1.0083 | Val Loss: 2.7123 | LR: 0.000500
Epoch 09 | Train Loss: 0.8919 | Val Loss: 2.7279 | LR: 0.000500
Epoch 10 | Train Loss: 0.8108 | Val Loss: 2.7595 | LR: 0.000500


Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(4932, 256)
    (lstm): LSTM(256, 512)
  )
  (decoder): Decoder(
    (attention): Attention(
      (attn): Linear(in_features=1024, out_features=512, bias=True)
      (v): Linear(in_features=512, out_features=1, bias=False)
    )
    (embedding): Embedding(13768, 256)
    (rnn): LSTM(768, 512)
    (fc_out): Linear(in_features=1280, out_features=13768, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [13]:
def translate_beam_search(sentence, model, eng_vocab, rus_vocab, device, beam_size=5, max_len=50, min_len=3):
    model.eval()
    if isinstance(sentence, str):
        tokens = sentence.lower().split()
    else:
        tokens = sentence

    src_indexes = [eng_vocab.get(t, eng_vocab["<oov>"]) for t in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device)

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src_tensor)

    beams = [([rus_vocab["<sos>"]], 0.0, hidden, cell)]

    for step in range(max_len):
        new_beams = []
        for seq, score, h, c in beams:
            last_token = seq[-1]
            if last_token == rus_vocab["<eos>"]:
                new_beams.append((seq, score, h, c))
                continue

            input_tensor = torch.LongTensor([last_token]).to(device)
            with torch.no_grad():
                output, h_new, c_new = model.decoder(input_tensor, h, c, encoder_outputs)

            probs = F.log_softmax(output, dim=1)
            if len(seq) < min_len:
                probs[0][rus_vocab["<eos>"]] = -1e9

            top_probs, top_idx = probs.topk(beam_size)
            for i in range(beam_size):
                token = top_idx[0][i].item()
                token_prob = top_probs[0][i].item()
                new_seq = seq + [token]
                new_score = score + token_prob
                new_beams.append((new_seq, new_score, h_new, c_new))

        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]
        if all(seq[-1] == rus_vocab["<eos>"] for seq, _, _, _ in beams):
            break

    best_seq = beams[0][0]
    inv_vocab = {v: k for k, v in rus_vocab.items()}

    result = []
    for idx in best_seq[1:]:
        word = inv_vocab[idx]
        if word == "<eos>":
            break
        result.append(word)

    return result

# --- BLEU Metric From Scratch ---
def get_ngrams(tokens, n):
    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngrams.append(tuple(tokens[i:i+n]))
    return Counter(ngrams)

def clipped_precision(hypothesis, reference, n):
    hyp_ngrams = get_ngrams(hypothesis, n)
    ref_ngrams = get_ngrams(reference, n)
    if not hyp_ngrams:
        return 0.0

    clipped_counts = 0
    total_hyp_ngrams = sum(hyp_ngrams.values())
    for ngram, count in hyp_ngrams.items():
        clipped_counts += min(count, ref_ngrams.get(ngram, 0))
    return clipped_counts / total_hyp_ngrams if total_hyp_ngrams > 0 else 0.0

def brevity_penalty(hypothesis, reference):
    hyp_len = len(hypothesis)
    ref_len = len(reference)
    if hyp_len == 0:
        return 0.0
    if hyp_len > ref_len:
        return 1.0
    return math.exp(1 - (ref_len / hyp_len))

def bleu_score(hypothesis_str, reference_str, max_n=4):
    hypothesis = [w for w in hypothesis_str.split() if w not in ['<sos>', '<eos>', '<pad>', '<oov>']]
    reference = [w for w in reference_str.split() if w not in ['<sos>', '<eos>', '<pad>', '<oov>']]

    bp = brevity_penalty(hypothesis, reference)
    log_precisions = 0.0
    weight = 1.0 / max_n

    for n in range(1, max_n + 1):
        p_n = clipped_precision(hypothesis, reference, n)
        if p_n == 0.0:
            return 0.0
        log_precisions += weight * math.log(p_n)

    return bp * math.exp(log_precisions)

In [14]:
# 1. Grab exactly 100 examples from our text validation split
sample_100 = val_text_pairs[:100]

def run_final_experiment(sample_pairs, model, eng_vocab, rus_vocab, device):
    results = []
    configs = [
        ("Greedy (k=1)", 1),
        ("Beam Search (k=3)", 3),
        ("Beam Search (k=5)", 5),
        ("Beam Search (k=10)", 10)
    ]

    print("Running 100-sentence evaluation... This will take a moment.")

    for config_name, k in configs:
        total_bleu4 = 0
        start_time = time.time()

        for eng_sent, rus_ref in sample_pairs:
            hyp_tokens = translate_beam_search(eng_sent, model, eng_vocab, rus_vocab, device, beam_size=k)
            hyp_str = " ".join(hyp_tokens)
            total_bleu4 += bleu_score(hyp_str, rus_ref, max_n=4)

        avg_time = (time.time() - start_time) / 100
        avg_bleu4 = total_bleu4 / 100

        results.append({
            "Decoding Strategy": config_name,
            "Average BLEU-4": round(avg_bleu4, 4),
            "Avg Time/Sentence (s)": round(avg_time, 4)
        })

    df = pd.DataFrame(results)
    print("\n" + "="*55)
    print("FINAL ASSIGNMENT RESULTS TABLE")
    print("="*55)
    print(df.to_string(index=False))
    print("="*55 + "\n")

    print("5 CONCRETE EXAMPLES (Greedy vs Beam k=5)")
    for i in range(10, 15):
        eng, ref = sample_pairs[i]
        clean_ref = " ".join([w for w in ref.split() if w not in ['<sos>', '<eos>']])

        greedy_hyp = " ".join(translate_beam_search(eng, model, eng_vocab, rus_vocab, device, beam_size=1))
        beam_hyp = " ".join(translate_beam_search(eng, model, eng_vocab, rus_vocab, device, beam_size=5))

        print(f"Example {i-9}:")
        print(f"Source: {eng}")
        print(f"Ref:    {clean_ref}")
        print(f"Greedy: {greedy_hyp}")
        print(f"Beam 5: {beam_hyp}")
        print("-" * 40)

# Run the final experiment
run_final_experiment(sample_100, model, eng_vocab, rus_vocab, device)

Running 100-sentence evaluation... This will take a moment.

FINAL ASSIGNMENT RESULTS TABLE
 Decoding Strategy  Average BLEU-4  Avg Time/Sentence (s)
      Greedy (k=1)          0.0267                 0.0146
 Beam Search (k=3)          0.0467                 0.0223
 Beam Search (k=5)          0.0467                 0.0248
Beam Search (k=10)          0.0467                 0.0454

5 CONCRETE EXAMPLES (Greedy vs Beam k=5)
Example 1:
Source: wheres tom from
Ref:    том откуда
Greedy: куда том том
Beam 5: куда том
----------------------------------------
Example 2:
Source: light the candle
Ref:    зажгите свечу
Greedy: зажги свечку
Beam 5: зажги свечку
----------------------------------------
Example 3:
Source: toms crazy
Ref:    том помешанный
Greedy: том ненормальный
Beam 5: том ненормальный
----------------------------------------
Example 4:
Source: i have it
Ref:    она у меня есть
Greedy: у меня
Beam 5: у меня
----------------------------------------
Example 5:
Source: i plead guilty
